In [6]:
import pandas as pd
import numpy as np

# 1. Загрузка данных
df = pd.read_csv('churn.csv')

print("=== Первые 5 строк датасета ===")
display(df.head())

print("\n=== Общая информация о датасете ===")
df.info()

=== Первые 5 строк датасета ===


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



=== Общая информация о датасете ===
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling

In [7]:
# 2. Проверка изначальных пропусков
print("=== Пропуски ДО обработки ===")
print(df.isnull().sum())

# 3. Искусственное создание пропусков (чтобы гарантированно показать работу fillna)
# Создаем 10 пропусков в числовом столбце
df.loc[0:9, 'MonthlyCharges'] = np.nan
# Создаем 10 пропусков в категориальном столбце
df.loc[10:19, 'InternetService'] = np.nan

# 4. Обработка "скрытых" пропусков в TotalCharges (пустые строки превращаем в NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("\n=== Пропуски после подготовки (для демонстрации) ===")
print(df[['MonthlyCharges', 'InternetService', 'TotalCharges']].isnull().sum())

=== Пропуски ДО обработки ===
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

=== Пропуски после подготовки (для демонстрации) ===
MonthlyCharges     10
InternetService    10
TotalCharges       11
dtype: int64


In [8]:
# Заполнение числового признака медианой (устойчива к выбросам)
monthly_median = df['MonthlyCharges'].median()
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(monthly_median)

# Заполнение категориального признака модой (самое частое значение)
internet_mode = df['InternetService'].mode()[0]
df['InternetService'] = df['InternetService'].fillna(internet_mode)

# Заполнение TotalCharges медианой
total_median = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(total_median)

print("=== Пропуски ПОСЛЕ заполнения (должны быть везде 0) ===")
print(df.isnull().sum())

=== Пропуски ПОСЛЕ заполнения (должны быть везде 0) ===
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Нормализуем все числовые столбцы (приводим к среднему=0 и отклонению=1)
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("=== Нормализованные числовые данные ===")
display(df[numeric_cols].head())

=== Нормализованные числовые данные ===


,tenure,MonthlyCharges,TotalCharges
0,-1.277445,0.185493,-0.994242
1,0.066327,0.185493,-0.173244
2,-1.236724,0.185493,-0.959674
3,0.514251,0.185493,-0.194766
4,-1.236724,0.185493,-0.940470


In [11]:
# Преобразуем все категориальные столбцы в бинарные
# drop_first=True УБИРАЕТ первый столбец категории, чтобы избежать мультиколлинеарности (переобучения)
df_processed = pd.get_dummies(df, drop_first=True)

print(f"=== Размерность датасета после OHE: {df_processed.shape} ===")
display(df_processed.head())

=== Размерность датасета после OHE: (7043, 7073) ===


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,customerID_0003-MKNFE,customerID_0004-TLHLJ,customerID_0011-IGKFF,customerID_0013-EXCHZ,customerID_0013-MHZWF,customerID_0013-SMEOE,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,-1.277445,0.185493,-0.994242,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,False
1,0,0.066327,0.185493,-0.173244,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,True,False
2,0,-1.236724,0.185493,-0.959674,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,True,True
3,0,0.514251,0.185493,-0.194766,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
4,0,-1.236724,0.185493,-0.940470,False,False,False,False,False,False,...,False,False,False,False,False,True,False,True,False,True


In [12]:
# Сохраняем обработанный датасет (index=False чтобы не сохранять номера строк)
df_processed.to_csv('processed_churn.csv', index=False)
print("✅ Файл processed_churn.csv успешно сохранён!")

✅ Файл processed_churn.csv успешно сохранён!
